# GuardCircle — Qwen3 7B Two-Stage SFT on SageMaker

## Training Strategy

```
Base Qwen3-8B
    │
    ▼  Stage 1: Cold Start (rule-based responses, ~50K+ public samples)
    │  • Broad scam pattern vocabulary (EN + ZH)
    │  • Learns output FORMAT and general fraud signals
    │  • Higher LR, more epochs — build strong foundation
    │
    ▼  Stage 2: Domain SFT (Sonnet-distilled, ~600 TW samples)
    │  • Taiwan-specific fraud patterns (貸款詐騙, 釣魚, 社群媒體)
    │  • Rich chain-of-thought justifications from Sonnet
    │  • Lower LR — fine-tune without forgetting Stage 1
    │
    ▼  GuardCircle Scam Detector (Qwen3-8B + LoRA)
```

## System Requirements
| Instance | VRAM | Use case |
|---|---|---|
| `ml.g5.2xlarge` | 24 GB (A10G) | Both stages, QLoRA 4-bit |
| `ml.g5.4xlarge` | 24 GB | Same GPU, more CPU/RAM for data loading |
| `ml.p3.2xlarge` | 16 GB (V100) | Fallback, reduce batch size |

## 0. Install Dependencies

In [ ]:
!pip install -q \
    transformers>=4.51.0 \
    datasets \
    accelerate \
    peft>=0.15.0 \
    trl>=0.16.0 \
    bitsandbytes \
    scipy \
    sagemaker \
    boto3

## 1. Configuration

In [ ]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-8B"

# ── Local data paths (from run_pipeline.py output) ────────────────────────────
S1_TRAIN = "./data/stage1_out/s1_train.jsonl"
S1_VAL   = "./data/stage1_out/s1_val.jsonl"
S2_TRAIN = "./data/stage2_out/s2_train.jsonl"
S2_VAL   = "./data/stage2_out/s2_val.jsonl"

# ── S3 ────────────────────────────────────────────────────────────────────────
S3_BUCKET = "your-sagemaker-bucket"          # <-- CHANGE THIS
S3_PREFIX = "guardcircle/sft"
S3_DATA   = f"s3://{S3_BUCKET}/{S3_PREFIX}/data"
S3_OUTPUT = f"s3://{S3_BUCKET}/{S3_PREFIX}/output"

# ── Stage 1 hyperparams (cold start — aggressive) ─────────────────────────────
S1_EPOCHS      = 2
S1_LR          = 3e-4
S1_BATCH       = 2
S1_GRAD_ACCUM  = 8    # effective batch = 16

# ── Stage 2 hyperparams (domain SFT — conservative) ──────────────────────────
S2_EPOCHS      = 3
S2_LR          = 1e-4  # lower LR to preserve Stage 1 knowledge
S2_BATCH       = 2
S2_GRAD_ACCUM  = 4    # effective batch = 8 (smaller dataset, smaller accum)

# ── Shared ─────────────────────────────────────────────────────────────────────
MAX_SEQ_LEN    = 2048
LORA_R         = 16
LORA_ALPHA     = 32
LORA_DROPOUT   = 0.05
LORA_TARGETS   = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

print(f"Model: {MODEL_ID}")
print(f"Stage 1: lr={S1_LR}, epochs={S1_EPOCHS}, eff_batch={S1_BATCH*S1_GRAD_ACCUM}")
print(f"Stage 2: lr={S2_LR}, epochs={S2_EPOCHS}, eff_batch={S2_BATCH*S2_GRAD_ACCUM}")

## 2. SageMaker Session + Upload Data

In [ ]:
import boto3
import sagemaker
from sagemaker.s3 import S3Uploader

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
print(f"Role: {role}")

# Upload all four data files
for local_path, label in [
    (S1_TRAIN, "Stage 1 Train"), (S1_VAL, "Stage 1 Val"),
    (S2_TRAIN, "Stage 2 Train"), (S2_VAL, "Stage 2 Val"),
]:
    if os.path.exists(local_path):
        uri = S3Uploader.upload(local_path, S3_DATA)
        print(f"✅ {label}: {uri}")
    else:
        print(f"⚠️  {label}: {local_path} not found — run run_pipeline.py first")

## 3. Inspect Datasets

In [ ]:
from datasets import load_dataset

def inspect(path, name):
    if not os.path.exists(path):
        print(f"⚠️  {name}: file not found")
        return
    ds = load_dataset("json", data_files=path, split="train")
    labels = {}
    sources = {}
    for r in ds:
        m = r.get("metadata", {})
        labels[m.get("label","?")] = labels.get(m.get("label","?"), 0) + 1
        sources[m.get("source","?")] = sources.get(m.get("source","?"), 0) + 1
    print(f"\n{name}: {len(ds)} samples")
    print(f"  labels:  {dict(sorted(labels.items()))}")
    print(f"  sources: {dict(sorted(sources.items()))}")
    sample = ds[0]["messages"]
    print(f"  user msg:  {sample[1]['content'][:80]}...")
    print(f"  asst resp: {sample[2]['content'][:80]}...")

inspect(S1_TRAIN, "Stage 1 Train")
inspect(S2_TRAIN, "Stage 2 Train")

## 4. Helper: Load Model with QLoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig


def load_qlora_model(base_model_id: str, lora_adapter_path: str = None):
    """Load base model in 4-bit, optionally resuming from a LoRA adapter."""
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        base_model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(base_model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    if lora_adapter_path:
        # Resume from previous LoRA checkpoint (Stage 1 → Stage 2)
        print(f"Loading LoRA adapter from: {lora_adapter_path}")
        model = PeftModel.from_pretrained(model, lora_adapter_path, is_trainable=True)
    else:
        # Fresh LoRA init
        model = prepare_model_for_kbit_training(model)
        lora_config = LoraConfig(
            r=LORA_R,
            lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            target_modules=LORA_TARGETS,
            bias="none",
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)

    model.print_trainable_parameters()
    return model, tokenizer


def format_dataset(ds, tokenizer):
    """Apply Qwen3 chat template to a HuggingFace dataset."""
    def _fmt(example):
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        return {"text": text}
    return ds.map(_fmt, remove_columns=ds.column_names)


def make_sft_config(output_dir, epochs, lr, batch, grad_accum, stage_label):
    return SFTConfig(
        output_dir=output_dir,
        run_name=stage_label,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch,
        per_device_eval_batch_size=batch,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        bf16=True,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        max_seq_length=MAX_SEQ_LEN,
        dataset_kwargs={"skip_prepare_dataset": True},
        report_to="none",
    )

print("Helpers loaded.")

## 5. Stage 1 — Cold Start Fine-tuning

Trains on `~50K+` public dataset samples with rule-based responses.  
Goal: learn the output format + broad scam pattern vocabulary.

In [ ]:
from datasets import load_dataset

S1_OUT_DIR = "./checkpoints/stage1"

# Load datasets
s1_train_ds = load_dataset("json", data_files=S1_TRAIN, split="train")
s1_val_ds   = load_dataset("json", data_files=S1_VAL,   split="train")
print(f"Stage 1 — Train: {len(s1_train_ds)}, Val: {len(s1_val_ds)}")

# Load model (fresh LoRA)
model, tokenizer = load_qlora_model(MODEL_ID)

# Format
s1_train_fmt = format_dataset(s1_train_ds, tokenizer)
s1_val_fmt   = format_dataset(s1_val_ds,   tokenizer)

# Train
s1_cfg = make_sft_config(S1_OUT_DIR, S1_EPOCHS, S1_LR, S1_BATCH, S1_GRAD_ACCUM, "stage1")
s1_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=s1_train_fmt, eval_dataset=s1_val_fmt,
    args=s1_cfg,
)

print("\n🚀 Starting Stage 1 training...")
s1_trainer.train()

# Save Stage 1 adapter
s1_trainer.save_model(S1_OUT_DIR)
tokenizer.save_pretrained(S1_OUT_DIR)
print(f"✅ Stage 1 complete — adapter saved to {S1_OUT_DIR}")

In [ ]:
# Upload Stage 1 adapter to S3
s1_s3 = S3Uploader.upload(S1_OUT_DIR, f"{S3_OUTPUT}/stage1-lora")
print(f"Stage 1 adapter uploaded: {s1_s3}")

## 6. Stage 2 — Domain SFT

Continues from the Stage 1 LoRA checkpoint.  
Trains on `~600` Taiwan-specific samples with Sonnet-generated justifications.  
Lower LR to preserve broad knowledge from Stage 1.

In [ ]:
S2_OUT_DIR = "./checkpoints/stage2"

# Load datasets
s2_train_ds = load_dataset("json", data_files=S2_TRAIN, split="train")
s2_val_ds   = load_dataset("json", data_files=S2_VAL,   split="train")
print(f"Stage 2 — Train: {len(s2_train_ds)}, Val: {len(s2_val_ds)}")

# Load model RESUMING from Stage 1 LoRA
# Free GPU memory first
import gc
del model, s1_trainer
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = load_qlora_model(MODEL_ID, lora_adapter_path=S1_OUT_DIR)

# Format
s2_train_fmt = format_dataset(s2_train_ds, tokenizer)
s2_val_fmt   = format_dataset(s2_val_ds,   tokenizer)

# Train
s2_cfg = make_sft_config(S2_OUT_DIR, S2_EPOCHS, S2_LR, S2_BATCH, S2_GRAD_ACCUM, "stage2")
s2_trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=s2_train_fmt, eval_dataset=s2_val_fmt,
    args=s2_cfg,
)

print("\n🚀 Starting Stage 2 training...")
s2_trainer.train()

# Save final adapter
s2_trainer.save_model(S2_OUT_DIR)
tokenizer.save_pretrained(S2_OUT_DIR)
print(f"✅ Stage 2 complete — final adapter saved to {S2_OUT_DIR}")

In [ ]:
# Upload final adapter to S3
s2_s3 = S3Uploader.upload(S2_OUT_DIR, f"{S3_OUTPUT}/stage2-lora-final")
print(f"Final adapter uploaded: {s2_s3}")

## 7. Inference Test

In [ ]:
SYSTEM_PROMPT = """你是一個專業的詐騙偵測助手。你的任務是分析用戶提供的訊息（可能是簡訊、網址、社群媒體貼文或其他文字內容），判斷其是否為詐騙，並提供詳細的分析理由。

請依照以下格式回覆：
1. **判定結果**：詐騙 / 非詐騙
2. **風險等級**：高風險 / 中風險 / 低風險
3. **詐騙類型**（若為詐騙）：例如貸款詐騙、釣魚網站、投資詐騙、購物詐騙等
4. **分析理由**：列出具體的可疑特徵或安全指標
5. **建議行動**：給用戶的具體建議"""


def detect_scam(text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"請分析以下訊息是否為詐騙，並提供完整的判斷理由：\n\n{text}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    model.eval()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=512,
            temperature=0.3, top_p=0.9, do_sample=True,
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


TEST_CASES = [
    # Stage 2 TW scam
    ("TW scam SMS",  "【中國信託】您的帳戶異常，系統更新將暫時關閉您的帳戶，在7/8前實名驗證，逾期者將關閉信用卡功能: https://ctbcbanklees.com/tw"),
    ("TW scam URL",  "https://einvoiceplatformd-tw.com/index/card.html"),
    ("TW loan scam", "《輕鬆借》30萬內當日撥款 免押免保 有工作來就借 0972326557 陳怡潔 線上諮詢 csms.tw/OXGHQ2"),
    # Stage 1 EN patterns
    ("EN phishing",  "Congratulations! You've been selected for a FREE iPhone 15. Click here to claim: bit.ly/free-iphone-scam"),
    ("EN spam SMS",  "WINNER!! You have won a 2-week FREE membership in our Price Reward Scheme! To claim call 08714712394"),
    # Legitimate
    ("TW legit SMS", "【中華電信】您的本期帳單金額為NT$499，繳費期限為2026/04/15，請至中華電信門市或網路繳費。"),
    ("Legit URL",    "https://www.shopee.tw/product/12345"),
]

for label, text in TEST_CASES:
    print(f"\n{'='*60}")
    print(f"[{label}] {text[:70]}..." if len(text) > 70 else f"[{label}] {text}")
    print(f"{'-'*60}")
    print(detect_scam(text))

## 8. (Optional) Merge LoRA & Export

Merge LoRA weights into the base model for standalone deployment (no PEFT dependency).

In [ ]:
MERGED_DIR = "./checkpoints/merged-final"

# Reload in fp16 for merging (can't merge from 4-bit)
del model, s2_trainer
gc.collect(); torch.cuda.empty_cache()

base_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16,
    device_map="auto", trust_remote_code=True,
)
merged = PeftModel.from_pretrained(base_fp16, S2_OUT_DIR)
merged = merged.merge_and_unload()

merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved: {MERGED_DIR}")

# Upload
merged_s3 = S3Uploader.upload(MERGED_DIR, f"{S3_OUTPUT}/merged-final")
print(f"Uploaded: {merged_s3}")

## 9. (Optional) Run as Managed SageMaker Training Job

Use this if you want to submit both stages as SageMaker training jobs instead of running inline.

In [ ]:
%%writefile training_scripts/train_two_stage.py
"""Two-stage SageMaker training script."""
import os, gc, json, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer, SFTConfig

def get_hp(key, default):
    return os.environ.get(f"SM_HP_{key.upper()}", str(default))

def main():
    data_dir   = os.environ.get("SM_CHANNEL_DATA", "/opt/ml/input/data/data")
    model_dir  = os.environ.get("SM_MODEL_DIR",   "/opt/ml/model")
    out_dir    = os.environ.get("SM_OUTPUT_DATA_DIR", "/opt/ml/output/data")

    model_id   = get_hp("MODEL_ID",  "Qwen/Qwen3-8B")
    max_seq    = int(get_hp("MAX_SEQ_LEN", 2048))
    lora_r     = int(get_hp("LORA_R",     16))
    lora_alpha = int(get_hp("LORA_ALPHA", 32))

    bnb = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
    )
    targets = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

    def load_base(adapter_path=None):
        m = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=bnb, device_map="auto", trust_remote_code=True)
        tok = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        if adapter_path:
            m = PeftModel.from_pretrained(m, adapter_path, is_trainable=True)
        else:
            m = prepare_model_for_kbit_training(m)
            m = get_peft_model(m, LoraConfig(
                r=lora_r, lora_alpha=lora_alpha, lora_dropout=0.05,
                target_modules=targets, bias="none", task_type="CAUSAL_LM"))
        m.print_trainable_parameters()
        return m, tok

    def fmt_ds(ds, tok):
        return ds.map(
            lambda ex: {"text": tok.apply_chat_template(
                ex["messages"], tokenize=False, add_generation_prompt=False)},
            remove_columns=ds.column_names)

    def train_stage(model, tok, train_ds, val_ds, cfg_kwargs, out_path):
        trainer = SFTTrainer(
            model=model, tokenizer=tok,
            train_dataset=fmt_ds(train_ds, tok),
            eval_dataset=fmt_ds(val_ds, tok),
            args=SFTConfig(
                output_dir=out_path,
                bf16=True, max_seq_length=max_seq,
                dataset_kwargs={"skip_prepare_dataset": True},
                report_to="none", **cfg_kwargs))
        trainer.train()
        trainer.save_model(out_path)
        tok.save_pretrained(out_path)
        return trainer

    # Stage 1
    s1_train = load_dataset("json", data_files=f"{data_dir}/s1_train.jsonl", split="train")
    s1_val   = load_dataset("json", data_files=f"{data_dir}/s1_val.jsonl",   split="train")
    print(f"Stage 1: train={len(s1_train)}, val={len(s1_val)}")
    m, tok = load_base()
    train_stage(m, tok, s1_train, s1_val, dict(
        num_train_epochs=int(get_hp("S1_EPOCHS", 2)),
        per_device_train_batch_size=int(get_hp("S1_BATCH", 2)),
        gradient_accumulation_steps=int(get_hp("S1_GRAD_ACCUM", 8)),
        learning_rate=float(get_hp("S1_LR", 3e-4)),
        lr_scheduler_type="cosine", warmup_ratio=0.05,
    ), f"{out_dir}/stage1")

    del m; gc.collect(); torch.cuda.empty_cache()

    # Stage 2
    s2_train = load_dataset("json", data_files=f"{data_dir}/s2_train.jsonl", split="train")
    s2_val   = load_dataset("json", data_files=f"{data_dir}/s2_val.jsonl",   split="train")
    print(f"Stage 2: train={len(s2_train)}, val={len(s2_val)}")
    m, tok = load_base(adapter_path=f"{out_dir}/stage1")
    train_stage(m, tok, s2_train, s2_val, dict(
        num_train_epochs=int(get_hp("S2_EPOCHS", 3)),
        per_device_train_batch_size=int(get_hp("S2_BATCH", 2)),
        gradient_accumulation_steps=int(get_hp("S2_GRAD_ACCUM", 4)),
        learning_rate=float(get_hp("S2_LR", 1e-4)),
        lr_scheduler_type="cosine", warmup_ratio=0.05,
    ), model_dir)   # final model → SM_MODEL_DIR (auto-uploaded to S3)

if __name__ == "__main__": main()

In [ ]:
from sagemaker.huggingface import HuggingFace
import os
os.makedirs("training_scripts", exist_ok=True)

estimator = HuggingFace(
    entry_point="train_two_stage.py",
    source_dir="./training_scripts",
    role=role,
    instance_type="ml.g5.2xlarge",
    instance_count=1,
    transformers_version="4.51.0",
    pytorch_version="2.6.0",
    py_version="py312",
    hyperparameters={
        "model_id":     MODEL_ID,
        "max_seq_len":  MAX_SEQ_LEN,
        "lora_r":       LORA_R,
        "lora_alpha":   LORA_ALPHA,
        "s1_epochs":    S1_EPOCHS,
        "s1_lr":        S1_LR,
        "s1_batch":     S1_BATCH,
        "s1_grad_accum":S1_GRAD_ACCUM,
        "s2_epochs":    S2_EPOCHS,
        "s2_lr":        S2_LR,
        "s2_batch":     S2_BATCH,
        "s2_grad_accum":S2_GRAD_ACCUM,
    },
    output_path=S3_OUTPUT,
    requirements=["peft>=0.15.0","trl>=0.16.0","bitsandbytes","scipy"],
)

estimator.fit({"data": S3_DATA})
print(f"\nJob: {estimator.latest_training_job.name}")
print(f"Model: {estimator.model_data}")